# STAT 764 · Lab 5 — The metric picks the winner

**Assigned Tue Oct 6 · Due Tue Oct 13, 11:59 pm**

Individual work. Part 1 needs Meeting 9 (Tuesday Oct 6); Parts 2 and 3 need Meeting 10
(Thursday Oct 8), and question **(d)** needs the reading, ISLP §4.3.

Same rules as before: `git pull`, copy into `work/` before editing, Restart & Run All before
submitting, upload the `.ipynb` to Canvas.

Meeting 9 said a classifier is a probability **and** a rule. This lab is about the numbers
you put on it afterward: **which metric you score with decides which model wins**, a
threshold priced by a decision beats one chosen by a metric, and a threshold you *tuned*
on your data flatters you the same way a tuned α did in Meeting 8.

⏱ The whole lab runs in under a minute on most laptops. Most of that is Part 3.

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:                       # you are inside (or just outside) your clone
    sys.path.insert(0, str(found[0] / "course"))
else:                           # Colab, or a copy saved outside the clone
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

from stat764 import diabetes

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, brier_score_loss,
                             recall_score, roc_auc_score)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Meeting 9's problem, exactly: readmitted within 30 days, with the encounters that ended in
# death or hospice removed -- those patients cannot be readmitted.
raw = diabetes()
ENDED = [11, 13, 14, 19, 20, 21]
d = raw[~raw["discharge_disposition_id"].isin(ENDED)].copy()
d["readmit30"] = (d["readmitted"] == "<30").astype(int)

NUM = ["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
       "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses"]
CAT = ["age", "gender", "race", "admission_type_id", "discharge_disposition_id",
       "admission_source_id", "A1Cresult", "max_glu_serum", "insulin", "change",
       "diabetesMed", "medical_specialty"]
d[CAT] = d[CAT].astype(str)            # the codes are labels, not quantities
PRIOR = ["number_inpatient", "number_emergency", "number_outpatient"]   # three visit counts


def readmission_model(num=NUM, cat=CAT, **logistic_args):
    """Meeting 9's pipeline. `num` and `cat` choose the columns; anything else goes to
    LogisticRegression -- e.g. readmission_model(class_weight="balanced")."""
    parts = [("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                               ("scale", StandardScaler())]), num)]
    if cat:
        parts.append(("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), cat))
    return Pipeline([("prep", ColumnTransformer(parts)),
                     ("model", LogisticRegression(max_iter=2000, **logistic_args))])


def split(seed):
    """Meeting 9's split: a quarter of the PATIENTS to the test set, each patient on one side."""
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed)
                  .split(d, groups=d["patient_nbr"]))
    return d.iloc[tr], d.iloc[te]


train, test = split(764)
y_train = train["readmit30"]
y_test = test["readmit30"].to_numpy()
print(f"train {len(train):,} encounters · test {len(test):,} encounters · "
      f"readmitted within 30 days in test: {y_test.mean():.1%}")

Everything in Parts 1 and 2 uses that one split, so your numbers will match mine. Your job
is the experiments, not the plumbing.

---

## Part 1 · Five metrics, four contestants (15 points)

**(i) Who wins?** Fit three models on `train` and score them on `test`:

| contestant | what it is |
|---|---|
| `baseline` | `readmission_model()` — Meeting 9's model |
| `balanced` | `readmission_model(class_weight="balanced")` — the usual advice for imbalanced data |
| `prior visits` | `readmission_model(PRIOR, [])` — only the three prior-visit counts |
| `base rate` | no model: the training readmission rate, `y_train.mean()`, as everyone's probability |

Report **one table**: for each contestant, accuracy at 0.5, recall at 0.5, AUC, average
precision, and the Brier score. Then name the winner under **each** metric.

Two of those are new, and each is one line:

- **average precision** (`average_precision_score`) — walk down the patients from highest
  probability to lowest and average the precision each time you reach a real readmission.
  It summarizes the whole ranking; higher is better.
- **Brier score** (`brier_score_loss`) — the average squared distance between each
  probability and what happened (0 or 1). It checks the probabilities themselves, not
  just their order; **lower** is better.

**(ii) The four boxes, by hand.** A team can review **10% of discharges**. For the
`baseline` and the `balanced` model, set each model's threshold at the 90th percentile of
*its own* test probabilities (`np.quantile(p, 0.90)`), so each flags the same number of
patients. For each, report the four boxes and compute **from them, by hand** —
not with a scikit-learn function — sensitivity, specificity, precision and negative
predictive value. Then check your sensitivity against `recall_score`.

⚠ Use the 90th percentile, not 0.90. The first is "the top 10% of patients"; the second is
a probability almost nobody reaches.

In [ ]:
# YOUR CODE HERE

## Part 2 · One curve for every price (20 points)

Meeting 10 priced one decision at a time. A **decision curve** prices all of them at once.

Choosing threshold *t* says: a patient is worth flagging once their chance of readmission
reaches *t*. At exactly *t* a flag breaks even, which prices a false alarm at
*t* / (1 − *t*) of a caught readmission. So count each caught readmission as **+1** and each
false alarm as **−*t* / (1 − *t*)**, per 100 patients — the **net benefit** of flagging at *t*:

In [ ]:
def net_benefit(y, p, t):
    """Net benefit per 100 patients of flagging everyone whose probability p is at least t.

    A caught readmission counts +1. A false alarm counts -t/(1-t): choosing t says a flag
    breaks even at probability t, which is exactly that exchange rate."""
    flag = p >= t
    caught = np.sum(flag & (y == 1))
    false_alarms = np.sum(flag & (y == 0))
    return 100 * (caught - false_alarms * t / (1 - t)) / len(y)


# Two strategies need no model at all -- every model has to beat both.
everyone = np.ones(len(y_test))                  # flag every patient
print(f"  flag everyone at t = 0.10: net benefit {net_benefit(y_test, everyone, 0.10):.2f} per 100")
print(f"  flag no one, at any t:     net benefit 0.00 per 100")

For every threshold from 0.01 to 0.50 in steps of 0.01, compute the net benefit of the
`baseline`, `prior visits` and `balanced` models from Part 1, and of flagging everyone.
Flagging no one is always 0. **Plot all five curves on one figure** and report:

1. the range of thresholds over which `baseline` beats **both** flagging everyone and
   flagging no one;
2. the largest gap between `baseline` and `prior visits`, and the threshold where it
   happens — and whether `prior visits` is ever the better of the two;
3. the largest threshold at which `balanced` still beats both;
4. each model's net benefit at Meeting 10's \$150 home visit that prevents 5% of
   readmissions: break-even `p_star = 150 / (0.05 * 17_700)`.

⚠ Compare with a small tolerance (`> 1e-9`). At the lowest thresholds every model flags
everyone, and two equal numbers can differ in the last decimal.

In [ ]:
# YOUR CODE HERE

## Part 3 · The threshold you tuned (15 points)

F1 picks a threshold by looking at the outcomes. So does any "best threshold" search. Here
is what that does to the number you then report. For each of 20 splits, fit the
`baseline` on that split's training patients and get probabilities for its test patients.
Then, at each size *n* = 250, 1,000 and 5,000, ten times:

- draw two **separate** sets of *n* test encounters, A and B;
- on A, find the threshold in `GRID` with the best F1, and record that F1 — **the number
  you would report**;
- on B, record the F1 at **that same** threshold — **the number you would get**.

Do the same for the fixed threshold `p_star` from Part 2 (no searching), recording its F1
on A and on B.

In [ ]:
GRID = np.round(np.arange(0.02, 0.61, 0.01), 2)      # thresholds to search


def f1_at(y, p, t):
    """F1 of flagging p >= t: 2 TP / (2 TP + FP + FN). Fast, and the same as f1_score."""
    flag = p >= t
    tp = np.sum(flag & (y == 1))
    return 2 * tp / (2 * tp + np.sum(flag & (y == 0)) + np.sum(~flag & (y == 1)))


# The shape of the loop -- you fill in the middle:
#
# for seed in range(20):
#     tr, te = split(seed)
#     p = readmission_model().fit(tr[NUM + CAT], tr["readmit30"]).predict_proba(te[NUM + CAT])[:, 1]
#     y = te["readmit30"].to_numpy()
#     rng = np.random.default_rng(seed)
#     for n in (250, 1000, 5000):
#         for rep in range(10):
#             order = rng.permutation(len(y))
#             A, B = order[:n], order[n:2 * n]          # two separate sets of n encounters
#             ...

Report **one table**. For each *n*: the median and the range (min to max) of the chosen
threshold; the median F1 on A at its chosen threshold; the median F1 on B at that same
threshold; and the median of **A minus B**. Then one more row per *n* for `p_star`: the
median of its A minus B.

⏱ Under a minute: twenty fits, and 600 small searches.

In [ ]:
# YOUR CODE HERE

## Part 4 · Written (25 points)

Three or four sentences each, and **every answer except (d) points at a number in your own
output.** Under each question, *A full answer* says what earns full credit — the same lines
are on the Canvas rubric.

**(a)** From Part 1. Which contestant does each metric crown? Why does accuracy at 0.5
crown a contestant that makes no use of any patient's information, and why is the
`balanced` model's Brier score worse than that contestant's? Then (ii): at 0.5 the two
models flag very different numbers of patients, yet at the top 10% their four boxes are
nearly the same. What did reweighting change, and what did it leave alone?

*A full answer:* names the winner under each of the five metrics from your table; explains
accuracy's choice through the base rate; explains the Brier score through what reweighting
did to the probabilities; and uses your two sets of four boxes to say that reweighting moved
the probabilities but barely moved the order.

**(b)** From Part 2. Over what range of thresholds does `baseline` earn its keep against
both no-model strategies? At the \$150 visit's break-even, which model would you use, and
what is it worth per 100 patients compared with the next best? Why does `balanced` fall
apart above about 0.1 when its AUC matches the baseline's?

*A full answer:* quotes the range from your curves; names the model and its net benefit at
`p_star`, with the comparison; and connects the `balanced` curve to Part 1 — a threshold
read as a probability needs probabilities that mean what they say.

**(c)** From Part 3. How much does reporting the F1 you tuned overstate the F1 you get, at
each size? Why does the overstatement shrink as *n* grows, and why is `p_star`'s gap about
zero? Name the Meeting 8 idea this is.

*A full answer:* quotes the median A − B at each *n* and the spread of chosen thresholds;
explains the shrinking with noise in small sets; says `p_star` was never chosen by looking
at outcomes; and names selection bias (the winner's curse).

**(d)** 📗 **From the reading.** ISLP §4.3.4 fits default on student status alone, then on
balance, income and student status together. The student coefficient changes sign. Explain
why, using the book's figure, and name the figure. Then: a bank must score one student
applicant. Which of the two models' answers should it use if it knows her credit-card
balance, and which if it does not?

*A full answer:* says why the sign flips (students carry higher balances, and at the same
balance they default less); names the figure; and answers both cases — the answer depends on
what the bank knows when it predicts. No number needed.

⚠ Answer (d) from the reading. It is there, and it is short.

**(e)** A hospital administrator emails: *"What is the best metric for our readmission
model?"* Write the three sentences you would send back, each naming a number from this lab.

*A full answer:* three sentences, each naming a number from this lab, that answer the
question the administrator should have asked.

### Your answers

*(double-click to edit)*

**(a)**

**(b)**

**(c)**

**(d)**

**(e)**

---

## Before you submit

- [ ] **Run → Restart Kernel and Run All Cells**
- [ ] Part 1 has the five-metric table with a winner per metric **and** both sets of four
      boxes with the four rates computed by hand
- [ ] Part 2 has the figure **and** the four things it asks for
- [ ] Part 3 has the table, including the `p_star` rows
- [ ] All five written answers are filled in, each pointing at a number
- [ ] Uploaded the `.ipynb` to Canvas

**75 points.**